# 01. AASIST-L Spoof Detection Training

This notebook trains the **AASIST-L** (Audio Anti-Spoofing using Integrated Spectro-Temporal Graph Attention Networks - Lightweight) on the **ASVspoof 2019 Logical Access (LA)** dataset.

### Pipeline Overview:
1. **Dataset Download & Preprocessing**: ASVspoof 2019 LA train/dev sets
2. **AASIST-L Architecture**: Sinc filterbank front-end → Graph Attention → Readout (~85k params)
3. **Loss Function**: Weighted Cross-Entropy / Angular Margin Loss
4. **Validation & Checkpoint Export**: Dev EER monitoring and exporting `checkpoints/AASIST-L.pth`

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

from backend.models.aasist import AASISTModel
from backend.data.loader import ASVspoofDataset
from backend.config import config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")

## 1. Load ASVspoof 2019 LA Dataset

In [ ]:
DATA_DIR = "./data/ASVspoof2019/LA"

# If data is present, initialize loaders
try:
    train_dataset = ASVspoofDataset(DATA_DIR, split="train", max_length=64600)
    dev_dataset = ASVspoofDataset(DATA_DIR, split="dev", max_length=64600)
    
    train_loader = DataLoader(train_dataset, batch_size=24, shuffle=True, num_workers=2)
    dev_loader = DataLoader(dev_dataset, batch_size=24, shuffle=False, num_workers=2)
    print(f"Train samples: {len(train_dataset)}, Dev samples: {len(dev_dataset)}")
except FileNotFoundError as e:
    print(f"Dataset not found at {DATA_DIR}. To download, see Edinburgh DataShare: https://datashare.ed.ac.uk/handle/10283/3336")

## 2. Model Initialization & Loss Setup

In [ ]:
model = AASISTModel().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"AASIST-L Trainable Parameters: {total_params:,}")

# Class weights (1.0 for bonafide [0], 9.0 for spoof [1] to handle class imbalance)
weights = torch.tensor([1.0, 9.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

## 3. Training Loop with EER Evaluation

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for batch in tqdm(loader, desc="Training"):
        x = batch["waveform"].to(device)
        y = batch["label"].to(device)
        
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

print("Training function configured. Ready to run over epochs.")

## 4. Export Pretrained Weights for Real-Time Inference Engine

In [ ]:
os.makedirs("./checkpoints", exist_ok=True)
checkpoint_path = "./checkpoints/AASIST-L.pth"
torch.save({"model": model.state_dict()}, checkpoint_path)
print(f"Saved AASIST-L checkpoint to {checkpoint_path}")